### ASOS dataset
ASOS weather refers to weather observations collected by the Automated Surface Observing System network

Dataset from:
https://mesonet.agron.iastate.edu/request/asos/1min.phtml

National Climatic Data Center is a part of the NOAA

In [113]:
import pandas as pd
import numpy as np

In [98]:
file = "NOAA data/ASOS_data_logan.csv"
df = pd.read_csv(file, low_memory=False)

In [99]:
df.head()

,station,station_name,valid(UTC),tmpf,dwpf,sknt,drct,gust_drct,gust_sknt,vis1_coeff,vis1_nd,vis2_coeff,vis2_nd,vis3_coeff,vis3_nd,ptype,precip,pres1,pres2,pres3
0,BOS,BOSTON/LOGAN INTL,1/1/2023 5:00,54,54,10,221,217,10,0.269,N,1.493,N,0.279,N,R-,0,29.492,29.493,29.494
1,BOS,BOSTON/LOGAN INTL,1/1/2023 5:05,54,54,10,221,216,11,0.173,N,0.283,N,0.226,N,R-,0,29.488,29.489,29.49
2,BOS,BOSTON/LOGAN INTL,1/1/2023 5:10,54,54,9,217,215,11,0.267,N,0.326,N,0.252,N,R-,0,29.48,29.482,29.482
3,BOS,BOSTON/LOGAN INTL,1/1/2023 5:15,54,54,8,212,211,9,0.38,N,0.414,N,0.424,N,R,0,29.471,29.472,29.473
4,BOS,BOSTON/LOGAN INTL,1/1/2023 5:20,54,54,9,207,211,11,0.231,N,0.359,N,0.34,N,R-,0,29.467,29.468,29.469


In [100]:
# Parse time column as UTC
df["timestamp_utc"] = pd.to_datetime(
    df["valid(UTC)"],
    errors="coerce",
    infer_datetime_format=True,
    utc=True,
)

C:\Users\USER\AppData\Local\Temp\ipykernel_14984\3026967470.py:2: UserWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, see https://pandas.pydata.org/pdeps/0004-consistent-to-datetime-parsing.html. You can safely remove this argument.
  df["timestamp_utc"] = pd.to_datetime(


In [101]:
# Convert extinction coefficients to numeric
ext_cols = ["vis1_coeff", "vis2_coeff", "vis3_coeff"]
for c in ext_cols:
    df[c] = pd.to_numeric(df[c], errors="coerce")

df = df.dropna(subset=["timestamp_utc"]).copy()
df.head()

,station,station_name,valid(UTC),tmpf,dwpf,sknt,drct,gust_drct,gust_sknt,vis1_coeff,...,vis2_coeff,vis2_nd,vis3_coeff,vis3_nd,ptype,precip,pres1,pres2,pres3,timestamp_utc
0,BOS,BOSTON/LOGAN INTL,1/1/2023 5:00,54,54,10,221,217,10,0.269,...,1.493,N,0.279,N,R-,0,29.492,29.493,29.494,2023-01-01 05:00:00+00:00
1,BOS,BOSTON/LOGAN INTL,1/1/2023 5:05,54,54,10,221,216,11,0.173,...,0.283,N,0.226,N,R-,0,29.488,29.489,29.49,2023-01-01 05:05:00+00:00
2,BOS,BOSTON/LOGAN INTL,1/1/2023 5:10,54,54,9,217,215,11,0.267,...,0.326,N,0.252,N,R-,0,29.48,29.482,29.482,2023-01-01 05:10:00+00:00
3,BOS,BOSTON/LOGAN INTL,1/1/2023 5:15,54,54,8,212,211,9,0.380,...,0.414,N,0.424,N,R,0,29.471,29.472,29.473,2023-01-01 05:15:00+00:00
4,BOS,BOSTON/LOGAN INTL,1/1/2023 5:20,54,54,9,207,211,11,0.231,...,0.359,N,0.340,N,R-,0,29.467,29.468,29.469,2023-01-01 05:20:00+00:00


In [102]:
# Constants
eps = 1e-9
MI_TO_KM = 1.60934

# ASOS daytime relationship: V (miles) = 3 / c
# We apply it to all records for simplicity.
df["vis1_km"] = 3.0 / df["vis1_coeff"].clip(lower=eps) * MI_TO_KM
df["vis2_km"] = 3.0 / df["vis2_coeff"].clip(lower=eps) * MI_TO_KM
df["vis3_km"] = 3.0 / df["vis3_coeff"].clip(lower=eps) * MI_TO_KM

# Mean extinction coefficient across the three sensors
df["ext_coeff_mean"] = df[ext_cols].mean(axis=1, skipna=True)

# Mean visibility in km across the three sensors
vis_km_cols = ["vis1_km", "vis2_km", "vis3_km"]
df["visibility_km_mean"] = df[vis_km_cols].mean(axis=1, skipna=True)

# Drop rows with no extinction or visibility info
df = df.dropna(subset=["ext_coeff_mean", "visibility_km_mean"]).copy()

df[["timestamp_utc"] + vis_km_cols + ["visibility_km_mean", "ext_coeff_mean"]].head()

df.head()

,station,station_name,valid(UTC),tmpf,dwpf,sknt,drct,gust_drct,gust_sknt,vis1_coeff,...,precip,pres1,pres2,pres3,timestamp_utc,vis1_km,vis2_km,vis3_km,ext_coeff_mean,visibility_km_mean
0,BOS,BOSTON/LOGAN INTL,1/1/2023 5:00,54,54,10,221,217,10,0.269,...,0,29.492,29.493,29.494,2023-01-01 05:00:00+00:00,17.948030,3.233771,17.304731,0.680333,12.828844
1,BOS,BOSTON/LOGAN INTL,1/1/2023 5:05,54,54,10,221,216,11,0.173,...,0,29.488,29.489,29.49,2023-01-01 05:05:00+00:00,27.907630,17.060141,21.362920,0.227333,22.110231
2,BOS,BOSTON/LOGAN INTL,1/1/2023 5:10,54,54,9,217,215,11,0.267,...,0,29.48,29.482,29.482,2023-01-01 05:10:00+00:00,18.082472,14.809877,19.158810,0.281667,17.350386
3,BOS,BOSTON/LOGAN INTL,1/1/2023 5:15,54,54,8,212,211,9,0.380,...,0,29.471,29.472,29.473,2023-01-01 05:15:00+00:00,12.705316,11.661884,11.386840,0.406000,11.918013
4,BOS,BOSTON/LOGAN INTL,1/1/2023 5:20,54,54,9,207,211,11,0.231,...,0,29.467,29.468,29.469,2023-01-01 05:20:00+00:00,20.900519,13.448524,14.200059,0.310000,16.183034


In [109]:
df["temp_ASOS"] = np.round((pd.to_numeric(df["tmpf"], errors="coerce") - 32) * 5 / 9, 2)
df["dewpoint_ASOS"] = np.round((pd.to_numeric(df["dwpf"], errors="coerce") - 32) * 5 / 9, 2)
df.head()

,station,station_name,valid(UTC),tmpf,dwpf,sknt,drct,gust_drct,gust_sknt,vis1_coeff,...,pres3,timestamp_utc,vis1_km,vis2_km,vis3_km,ext_coeff_mean,visibility_km_mean,tmpc_ASOS,temp_ASOS,dewpoint_ASOS
0,BOS,BOSTON/LOGAN INTL,1/1/2023 5:00,54,54,10,221,217,10,0.269,...,29.494,2023-01-01 05:00:00+00:00,17.948030,3.233771,17.304731,0.680333,12.828844,12.22,12.22,12.22
1,BOS,BOSTON/LOGAN INTL,1/1/2023 5:05,54,54,10,221,216,11,0.173,...,29.49,2023-01-01 05:05:00+00:00,27.907630,17.060141,21.362920,0.227333,22.110231,12.22,12.22,12.22
2,BOS,BOSTON/LOGAN INTL,1/1/2023 5:10,54,54,9,217,215,11,0.267,...,29.482,2023-01-01 05:10:00+00:00,18.082472,14.809877,19.158810,0.281667,17.350386,12.22,12.22,12.22
3,BOS,BOSTON/LOGAN INTL,1/1/2023 5:15,54,54,8,212,211,9,0.380,...,29.473,2023-01-01 05:15:00+00:00,12.705316,11.661884,11.386840,0.406000,11.918013,12.22,12.22,12.22
4,BOS,BOSTON/LOGAN INTL,1/1/2023 5:20,54,54,9,207,211,11,0.231,...,29.469,2023-01-01 05:20:00+00:00,20.900519,13.448524,14.200059,0.310000,16.183034,12.22,12.22,12.22


In [110]:
# Keep only relevant columns
asos_clean = df[[
    "timestamp_utc",
    "tmpc_ASOS",
    "dewpoint_ASOS",
    "vis1_coeff", "vis2_coeff", "vis3_coeff",
    "ext_coeff_mean",
    "vis1_km", "vis2_km", "vis3_km",
    "visibility_km_mean",
]].sort_values("timestamp_utc")

asos_clean.head()

,timestamp_utc,tmpc_ASOS,dewpoint_ASOS,vis1_coeff,vis2_coeff,vis3_coeff,ext_coeff_mean,vis1_km,vis2_km,vis3_km,visibility_km_mean
0,2023-01-01 05:00:00+00:00,12.22,12.22,0.269,1.493,0.279,0.680333,17.948030,3.233771,17.304731,12.828844
1,2023-01-01 05:05:00+00:00,12.22,12.22,0.173,0.283,0.226,0.227333,27.907630,17.060141,21.362920,22.110231
2,2023-01-01 05:10:00+00:00,12.22,12.22,0.267,0.326,0.252,0.281667,18.082472,14.809877,19.158810,17.350386
3,2023-01-01 05:15:00+00:00,12.22,12.22,0.380,0.414,0.424,0.406000,12.705316,11.661884,11.386840,11.918013
4,2023-01-01 05:20:00+00:00,12.22,12.22,0.231,0.359,0.340,0.310000,20.900519,13.448524,14.200059,16.183034


In [111]:
# Compare this result during foggy time with:
# https://www.weather.gov/wrh/timeseries?site=KBOS&hours=48&units=english&chart=on&headers=on&obs=tabular&hourly=true&pview=standard&font=12&history=yes&start=20251101&end=20251130&plot=
start = pd.Timestamp("2025-11-10 6:00", tz="UTC")
end = pd.Timestamp("2025-11-10 10:00", tz="UTC")

asos_clean.loc[asos_clean["timestamp_utc"].between(start, end)]

,timestamp_utc,tmpc_ASOS,dewpoint_ASOS,vis1_coeff,vis2_coeff,vis3_coeff,ext_coeff_mean,vis1_km,vis2_km,vis3_km,visibility_km_mean
255756,2025-11-10 06:40:00+00:00,13.33,12.78,0.627,0.529,2.271,1.142333,7.700191,9.126692,2.125945,6.317609
255757,2025-11-10 06:45:00+00:00,12.78,12.78,0.633,0.755,2.565,1.317667,7.627204,6.394728,1.882269,5.301400
255758,2025-11-10 06:50:00+00:00,12.78,12.78,0.602,0.851,2.222,1.225000,8.019967,5.673349,2.172826,5.288714
255759,2025-11-10 06:55:00+00:00,12.78,12.22,0.540,1.143,2.510,1.397667,8.940778,4.223990,1.923514,5.029427
255760,2025-11-10 07:00:00+00:00,12.78,12.22,0.899,1.626,3.436,1.987000,5.370434,2.969262,1.405128,3.248275
255761,2025-11-10 07:05:00+00:00,12.78,12.22,0.760,1.747,4.503,2.336667,6.352658,2.763606,1.072179,3.396148
255762,2025-11-10 07:10:00+00:00,12.78,12.22,2.011,2.346,6.406,3.587667,2.400806,2.057980,0.753672,1.737486
255763,2025-11-10 07:15:00+00:00,12.78,12.22,1.213,1.971,5.903,3.029000,3.980231,2.449528,0.817893,2.415884
255764,2025-11-10 07:20:00+00:00,12.78,12.22,2.840,2.350,6.730,3.973333,1.700007,2.054477,0.717388,1.490624
255765,2025-11-10 07:25:00+00:00,12.22,11.67,4.167,4.411,2.294,3.624000,1.158632,1.094541,2.104629,1.452601


In [112]:
# Save cleaned ASOS visibility dataset
asos_clean.to_csv("Filtered dataset/ASOS_visibility_5min_clean.csv", index=False)